# Lab 5: Regularisation, Diagnostics, and Convolutional Neural Networks

**DATA425 | Foundations of Deep Learning**

## What this lab is about

This lab connects two important parts of the course.

First, we practise **diagnostics and regularisation**. We use training and validation curves to decide whether a model is underfitting, overfitting, or generalising reasonably well.

Second, we introduce **convolutional neural networks (CNNs)**. CNNs are designed for data with spatial structure, especially images. Instead of treating every pixel as unrelated, a CNN looks for local patterns such as edges, corners, strokes, and textures.

By the end, you should be able to explain why CNNs are useful for images and how validation curves help guide modelling decisions.


## 0. Setup

The notebook uses NumPy, Matplotlib, scikit-learn, and PyTorch. PyTorch is used here because it gives us a clear view of the training loop while still letting us build neural network layers easily.

The code automatically uses a GPU if one is available. If not, it uses the CPU. The dataset is small enough that either option should work.


In [ ]:
import importlib.util
import subprocess
import sys

REQUIRED = {
    "numpy": "numpy",
    "matplotlib": "matplotlib",
    "sklearn": "scikit-learn",
    "torch": "torch"
}

missing = [package for module, package in REQUIRED.items()
           if importlib.util.find_spec(module) is None]

if missing:
    print("Installing missing packages:", ", ".join(missing))
    try:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)
    except subprocess.CalledProcessError as exc:
        raise RuntimeError(
            "Automatic installation failed. Install these packages manually or run this notebook "
            "in an environment with internet access: " + ", ".join(missing)
        ) from exc
else:
    print("All required packages are already installed.")

In [ ]:
import random
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from sklearn.preprocessing import StandardScaler

SEED = 425
np.random.seed(SEED)
random.seed(SEED)
torch.manual_seed(SEED)
torch.set_num_threads(1)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", DEVICE)

plt.rcParams["figure.figsize"] = (8, 5)
plt.rcParams["font.size"] = 12
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.25

The setup prints the device being used. If you see `cpu`, that is fine for this lab because the models and datasets are intentionally small.


## 1. Diagnostics: underfitting and overfitting

A model is useful when it performs well on new data, not only on the examples it trained on.

Training and validation curves help us diagnose what is happening:

- **Underfitting:** both training and validation losses are high. The model is too simple, not trained enough, or missing useful features.
- **Good fit:** training and validation losses both decrease and stay reasonably close.
- **Overfitting:** training loss keeps decreasing, but validation loss stops improving or gets worse.

We start with noisy regression data because it gives a simple setting for seeing these curve patterns.


In [ ]:
def make_regression_data(n=180, seed=SEED):
    rng = np.random.default_rng(seed)
    X = rng.uniform(-3, 3, size=(n, 3)).astype("float32")
    y = (np.sin(X[:, 0]) + 0.5 * X[:, 1] ** 2 - 0.4 * X[:, 2]
         + rng.normal(0, 0.45, size=n)).astype("float32")
    return X, y.reshape(-1, 1)

X_reg, y_reg = make_regression_data()
X_train, X_val, y_train, y_val = train_test_split(X_reg, y_reg, test_size=0.35, random_state=SEED)

scaler = StandardScaler().fit(X_train)
X_train_s = scaler.transform(X_train).astype("float32")
X_val_s = scaler.transform(X_val).astype("float32")

plt.scatter(X_train[:, 0], y_train, label="training", alpha=0.7, edgecolors="k", linewidth=0.3)
plt.scatter(X_val[:, 0], y_val, label="validation", alpha=0.7, edgecolors="k", linewidth=0.3)
plt.xlabel("feature 1")
plt.ylabel("target")
plt.title("Noisy regression data")
plt.legend()
plt.show()

The data contains noise, so no model should fit every point perfectly. If a model tries too hard to match every noisy detail, validation performance can suffer.


## 2. Training loop helper

The next cell defines a reusable PyTorch training loop. The loop follows the same structure we have used throughout the course:

1. Put the model in training mode.
2. Make predictions for a mini-batch.
3. Compute the loss.
4. Backpropagate the gradients.
5. Update the parameters.
6. Evaluate on validation data without updating weights.

The function returns a history dictionary so we can plot training and validation curves later.


In [ ]:
def make_loader(X, y, batch_size=32, shuffle=True):
    X_tensor = torch.tensor(X, dtype=torch.float32)
    y_tensor = torch.tensor(y)
    dataset = TensorDataset(X_tensor, y_tensor)
    generator = torch.Generator().manual_seed(SEED)
    return DataLoader(dataset, batch_size=batch_size, shuffle=shuffle, generator=generator)


def train_model(model, train_loader, val_loader, loss_fn, optimizer, epochs=30, task="regression"):
    model.to(DEVICE)
    history = {"train_loss": [], "val_loss": [], "val_accuracy": []}
    for epoch in range(epochs):
        model.train()
        train_losses = []
        for xb, yb in train_loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            optimizer.zero_grad()
            outputs = model(xb)
            loss = loss_fn(outputs, yb)
            loss.backward()
            optimizer.step()
            train_losses.append(loss.item())

        model.eval()
        val_losses = []
        preds, targets = [], []
        with torch.no_grad():
            for xb, yb in val_loader:
                xb, yb = xb.to(DEVICE), yb.to(DEVICE)
                outputs = model(xb)
                val_losses.append(loss_fn(outputs, yb).item())
                if task == "classification":
                    preds.append(outputs.argmax(dim=1).cpu().numpy())
                    targets.append(yb.cpu().numpy())

        history["train_loss"].append(float(np.mean(train_losses)))
        history["val_loss"].append(float(np.mean(val_losses)))
        if task == "classification":
            preds = np.concatenate(preds)
            targets = np.concatenate(targets)
            history["val_accuracy"].append(float(accuracy_score(targets, preds)))
    return history


def plot_loss_curves(histories, title="Training curves"):
    for name, hist in histories.items():
        plt.plot(hist["train_loss"], linestyle="--", label=f"{name} train")
        plt.plot(hist["val_loss"], label=f"{name} validation")
    plt.xlabel("epoch")
    plt.ylabel("loss")
    plt.title(title)
    plt.legend()
    plt.show()

Notice the difference between `model.train()` and `model.eval()`. Layers such as dropout behave differently during training and evaluation, so switching modes is important.


## 3. Overfitting and regularisation in a regression model

We train two networks on the same data:

- a large model with no regularisation;
- a regularised model with dropout and weight decay.

**Dropout** randomly switches off some hidden activations during training. This makes it harder for the model to rely too heavily on any one pathway.

**Weight decay** is the PyTorch name for L2 regularisation. It discourages very large weights, which can reduce overly complex fits.


In [ ]:
class RegressionMLP(nn.Module):
    def __init__(self, dropout=0.0):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(3, 128), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(128, 128), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(128, 64), nn.ReLU(),
            nn.Linear(64, 1),
        )
    def forward(self, x):
        return self.net(x)

train_loader = make_loader(X_train_s, y_train, batch_size=24, shuffle=True)
val_loader = make_loader(X_val_s, y_val, batch_size=64, shuffle=False)

loss_fn = nn.MSELoss()

torch.manual_seed(SEED)
overfit_model = RegressionMLP(dropout=0.0)
overfit_history = train_model(
    overfit_model,
    train_loader,
    val_loader,
    loss_fn,
    torch.optim.Adam(overfit_model.parameters(), lr=0.01),
    epochs=50,
)

torch.manual_seed(SEED)
regularised_model = RegressionMLP(dropout=0.25)
regularised_history = train_model(
    regularised_model,
    train_loader,
    val_loader,
    loss_fn,
    torch.optim.Adam(regularised_model.parameters(), lr=0.01, weight_decay=1e-3),
    epochs=50,
)

plot_loss_curves({"overfit": overfit_history, "regularised": regularised_history},
                 "Regularisation changes the validation curve")

Compare the training and validation curves. Regularisation usually increases training loss slightly because it makes the task harder, but it can improve validation behaviour by reducing memorisation.


## 4. Why convolution helps with images

A fully connected network flattens an image into a long vector. Once the image is flattened, nearby pixels are no longer treated as nearby by the model.

A convolutional network keeps the two-dimensional structure. It uses small filters that slide across the image. This gives CNNs two useful properties:

- **local pattern detection:** filters can learn small patterns such as edges and strokes;
- **parameter sharing:** the same filter is used across the image, so the model can detect a pattern wherever it appears.

Pooling layers reduce spatial size and help the model focus on whether a pattern exists rather than its exact pixel location.


In [ ]:
digits = load_digits()
X_img = (digits.images.astype("float32") / 16.0)[:, None, :, :]
y_img = digits.target.astype("int64")

X_train_img, X_test_img, y_train_img, y_test_img = train_test_split(
    X_img, y_img, test_size=0.25, random_state=SEED, stratify=y_img
)
X_train_img, X_val_img, y_train_img, y_val_img = train_test_split(
    X_train_img, y_train_img, test_size=0.20, random_state=SEED, stratify=y_train_img
)

fig, axes = plt.subplots(2, 5, figsize=(8, 4))
for ax, image, label in zip(axes.ravel(), X_train_img[:10], y_train_img[:10]):
    ax.imshow(image[0], cmap="gray")
    ax.set_title(f"label {label}")
    ax.axis("off")
plt.tight_layout()
plt.show()

train_img_loader = make_loader(X_train_img, y_train_img.astype("int64"), batch_size=64, shuffle=True)
val_img_loader = make_loader(X_val_img, y_val_img.astype("int64"), batch_size=128, shuffle=False)
test_img_loader = make_loader(X_test_img, y_test_img.astype("int64"), batch_size=128, shuffle=False)

The digit images are only 8 by 8 pixels, but they still have spatial structure. The shape of each stroke matters, so a CNN has a useful inductive bias for this task.


## 5. Compare an MLP and a small CNN

We now compare two image classifiers:

- **MLP:** flattens the image and uses dense layers.
- **CNN:** keeps the image shape and learns convolutional filters.

Both models are small so the comparison runs quickly. The point is not to claim that the CNN will always win on every tiny dataset. The point is to see how the architecture changes the way the model uses the image.


In [ ]:
class ImageMLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64, 64), nn.ReLU(),
            nn.Linear(64, 10),
        )
    def forward(self, x):
        return self.net(x)


class SmallCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(1, 8, kernel_size=3, padding=1), nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(8, 16, kernel_size=3, padding=1), nn.ReLU(),
            nn.Flatten(),
            nn.Linear(16 * 4 * 4, 32), nn.ReLU(),
            nn.Linear(32, 10),
        )
    def forward(self, x):
        return self.net(x)


def evaluate_classifier(model, loader):
    model.eval()
    preds, targets = [], []
    with torch.no_grad():
        for xb, yb in loader:
            xb = xb.to(DEVICE)
            outputs = model(xb)
            preds.append(outputs.argmax(dim=1).cpu().numpy())
            targets.append(yb.numpy())
    preds = np.concatenate(preds)
    targets = np.concatenate(targets)
    return accuracy_score(targets, preds), preds, targets

histories = {}
models = {}
for name, model in {"MLP": ImageMLP(), "CNN": SmallCNN()}.items():
    torch.manual_seed(SEED)
    optimizer = torch.optim.Adam(model.parameters(), lr=0.003, weight_decay=1e-4)
    history = train_model(model, train_img_loader, val_img_loader,
                          nn.CrossEntropyLoss(), optimizer, epochs=8, task="classification")
    acc, _, _ = evaluate_classifier(model.to(DEVICE), test_img_loader)
    histories[name] = history
    models[name] = model
    print(f"{name} test accuracy: {acc:.3f}")

plt.plot(histories["MLP"]["val_accuracy"], label="MLP validation accuracy")
plt.plot(histories["CNN"]["val_accuracy"], label="CNN validation accuracy")
plt.xlabel("epoch")
plt.ylabel("validation accuracy")
plt.title("Validation accuracy during training")
plt.legend()
plt.show()

Compare validation accuracy over epochs as well as final test accuracy. The training path often tells you more than a single final number.


## 6. Inspect CNN filters and errors

The first convolutional layer learns small 3 by 3 filters. On large image datasets, early filters often become edge or texture detectors. On this tiny digits dataset, the filters are simple, but they still show that the CNN is learning reusable local patterns.

We also print a confusion matrix and classification report. These help answer more detailed questions than accuracy alone:

- Which digits are confused with each other?
- Are some classes much harder than others?
- Is the model making one type of mistake repeatedly?


In [ ]:
cnn = models["CNN"].to("cpu")
first_layer = cnn.net[0]
filters = first_layer.weight.detach().numpy()

fig, axes = plt.subplots(2, 4, figsize=(7, 4))
for ax, filt in zip(axes.ravel(), filters[:8]):
    ax.imshow(filt[0], cmap="gray")
    ax.axis("off")
plt.suptitle("First-layer CNN filters")
plt.tight_layout()
plt.show()

cnn.to(DEVICE)
acc, preds, targets = evaluate_classifier(cnn, test_img_loader)
print("CNN test accuracy:", f"{acc:.3f}")
print("Confusion matrix:")
print(confusion_matrix(targets, preds))
print(classification_report(targets, preds))

Use the confusion matrix to look for patterns in the mistakes. For example, some handwritten digits naturally look similar, so errors may not be random.


## Your turn

Try one small experiment and explain what changed.

Suggested experiments:

1. Increase the dropout in the regression model and compare the validation curve.
2. Remove `MaxPool2d` from the CNN and see whether training changes.
3. Add one more convolutional layer to the CNN.
4. Increase the number of CNN filters from 8 and 16 to larger values.

Change one thing at a time. That makes it easier to connect the result to the modelling idea.


In [ ]:
# Optional exercise scaffold.
# Add your experiment here. This cell is safe to run as-is.
pass

## Summary

In this lab, you practised two sets of tools:

- Diagnostics: training and validation curves help identify underfitting, overfitting, and reasonable generalisation.
- Regularisation: dropout and weight decay can reduce overfitting by limiting memorisation.
- CNNs: convolutional layers use local filters and parameter sharing, which makes them well suited to images.
- Error analysis: confusion matrices help reveal which classes the model struggles with.

The main takeaway is that modelling is not only about getting a high accuracy number. We also need to understand how the model learned, where it fails, and whether its architecture matches the structure of the data.
